## Daten laden


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%reload_ext autoreload

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from utils import bewerten, trainieren_und_bewerten

df = pd.read_parquet('../data/processed/credit_transformed.parquet')
df.dtypes

---
## Merkmale trennen und Kategorien in Zahlen umwandeln

In [ ]:
X = df.drop(columns='loan_status')
y = df['loan_status'].astype(int)

X = pd.get_dummies(X, columns=['person_home_ownership', 'loan_intent'], drop_first=True, dtype=float)

X.head()

In [ ]:
y.head()

---
## Train-Test Split 80/20

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

---
## Trainings Mediane abgrenzen
Sonst würde ein Data-leakage ein zu gutes Ergebnis aka overfitting erzeugen
Eigentlich gehört dieser Schritt in 02_Datapreparation aber da es einen Mergeconflict mit Carl geben könnte, ist es besser hier aufgehoben.

Der Gedanke hinter dem Median: Die Daten zu löschen würde uns 10% des Datensatzes kosten. Wenn wir den Median in die N.A werte einfüllen, dann gibt es keine Ausreißer und wir können die Daten trotzdem noch verwenden.

Man könnte auch noch ein Schätzungsmodell für die Daten bauen aber das ist für die paar daten zu aufwendig..

In [ ]:
mediane = X_train.median()
X_train = X_train.fillna(mediane)
X_test = X_test.fillna(mediane)

X_train.isna().sum().sum(), X_test.isna().sum().sum()   # sollte (0, 0) ergeben

---
## Daten speichern

In [ ]:
from pathlib import Path

ordner = Path('../data/processed')
ordner.mkdir(parents=True, exist_ok=True)

X_train.to_parquet(ordner / 'X_train.parquet')
X_test.to_parquet(ordner / 'X_test.parquet')
y_train.to_frame().to_parquet(ordner / 'y_train.parquet')
y_test.to_frame().to_parquet(ordner / 'y_test.parquet')

---
## Modellvergleiche 


Statt alle Verfahren gleichwertig zu testen, schränken wir die Auswahl vorab anhand der Literatur und der Anforderungen der Bank ein. Grundlage ist vor allem Lessmann et al. (2015), die 41 Verfahren auf Datensätzen aus dem Privatkundenkredit vergleichen und damit sehr nah an unserem Anwendungsfall sind.

| Modell | Rang bei Lessmann et al. | Entscheidung | Begründung |
|---|---|---|---|
| Random Forest | 4,46 | behalten | In beiden Studien bestes Verfahren, stabil über alle Bewertungsmaße |
| Gradient Boosting | 8,34 / 9,88 | behalten | Im oberen Feld, moderne Implementierungen laut Grinsztajn et al. (2022) gleichauf mit Random Forest, verarbeitet fehlende Werte selbst |
| Logistische Regression | 9,34 | behalten (Baseline) | Branchenstandard, gut erklärbar, liefert die besten Wahrscheinlichkeiten (Brier Score) |
| Neuronales Netz | 8,73 | ausgeschlossen | Genauigkeit ähnlich wie logistische Regression, aber nicht erklärbar und schwache Wahrscheinlichkeiten |
| SVC | 12,08 | ausgeschlossen | Mittelfeld, hoher Rechenaufwand bei unserer Datenmenge, nicht erklärbar |
| kNN | 16,38 | ausgeschlossen | Schwach, empfindlich gegenüber Ausreißern und unwichtigen Merkmalen |
| Naive Bayes | 16,92 | ausgeschlossen | Schwach, Unabhängigkeitsannahme passt nicht zu unseren Daten |
| Entscheidungsbaum | 20,70 | ausgeschlossen | Schwächstes Verfahren, wird durch Random Forest und Boosting abgelöst |
| Lineare Regression | – | ausgeschlossen | Kein Klassifikationsverfahren, Ausgaben außerhalb von 0 und 1 |

Rang = durchschnittlicher Rang über vier Bewertungsmaße (AUC, Accuracy, Brier Score, H-Measure), kleiner ist besser. Bei Gradient Boosting stehen zwei Werte, weil die Studie AdaBoost und Stochastic Gradient Boosting getrennt bewertet.

Das neuronale Netz liegt knapp vor der logistischen Regression. Wir behalten trotzdem die logistische Regression, weil Kreditentscheidungen nachvollziehbar sein müssen: Die Kreditwürdigkeitsprüfung gilt im EU AI Act als Hochrisiko-Anwendung, und Rudin (2019) empfiehlt bei solchen Entscheidungen erklärbare Modelle.

Die ausgeschlossenen Modelle haben wir zur Kontrolle trotzdem trainiert. Die Ergebnisse stehen im Anhang am Ende des Notebooks und bestätigen die Vorauswahl.

**Quellen**
- Lessmann, S., Baesens, B., Seow, H.-V. & Thomas, L. C. (2015). Benchmarking state-of-the-art classification algorithms for credit scoring: An update of research. *European Journal of Operational Research*, 247(1), 124–136.
- Rudin, C. (2019). Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead. *Nature Machine Intelligence*, 1, 206–215.

---
### 1. Logistic Regression

In [ ]:
model_log = trainieren_und_bewerten('Logistische Regression', make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=1000)), X_train, y_train, X_test, y_test)

---
### 2. Random Forest

In [ ]:
model_rf =trainieren_und_bewerten('Random Forest', RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42), X_train, y_train, X_test, y_test)

---
### 3. Gradient Boosting

In [ ]:
model_gb = trainieren_und_bewerten('Gradient Boosting', HistGradientBoostingClassifier(random_state=42),
                                   X_train, y_train, X_test, y_test)

---
###
